# 01 · Exploratory data analysis

**Project:** Machine Learning Based Algorithmic Trading Signals: A Comparative Study Against Technical Baseline Strategies

This notebook looks at the raw market data before any modelling: coverage, return distributions, autocorrelation
(how predictable daily returns are) and the distribution of the three-class target. All data is loaded through the
same `src.data_loader` used by the pipeline, from the local cache in `data/raw/` (it is downloaded on the first run).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.utils import load_config
cfg = load_config(ROOT / "config.yaml")
RESULTS = ROOT / cfg["experiment"]["results_dir"]

In [ ]:
from src.data_loader import load_universe
frames, manifest = load_universe(cfg)
pd.DataFrame(manifest)[["ticker", "source", "is_synthetic", "rows", "first_date", "last_date"]]

## Data validation report
Every cleaning action is counted; nothing is imputed.

In [ ]:
pd.DataFrame({m["ticker"]: m["validation"] for m in manifest})

## Summary statistics of daily close-to-close returns

In [ ]:
rows = []
for t, df in frames.items():
    r = df["close"].pct_change().dropna()
    rows.append({"ticker": t, "days": len(r), "mean_daily_%": r.mean() * 100, "std_daily_%": r.std() * 100,
                 "annualised_return_%": ((1 + r).prod() ** (252 / len(r)) - 1) * 100,
                 "annualised_vol_%": r.std() * np.sqrt(252) * 100, "skew": r.skew(), "excess_kurtosis": r.kurt(),
                 "share_up_days_%": (r > 0).mean() * 100})
pd.DataFrame(rows).round(3)

## Adjusted price history (log scale)
Shaded regions are the validation and test periods used by the holdout experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
for t, df in frames.items():
    ax.plot(df.index, df["close"] / df["close"].iloc[0], label=t, lw=1.2)
ax.set_yscale("log"); ax.set_ylabel("Growth of 1 USD (log scale)")
ax.axvspan(pd.Timestamp(cfg["split"]["train_end"]), pd.Timestamp(cfg["split"]["val_end"]), color="orange", alpha=0.12, label="validation")
ax.axvspan(pd.Timestamp(cfg["split"]["val_end"]), df.index[-1], color="green", alpha=0.10, label="test")
ax.legend(); ax.set_title("Adjusted close, normalised to 1 at the first date"); plt.show()

## Return distribution
Daily returns have fatter tails (higher kurtosis) than a normal distribution with the same variance.

In [ ]:
fig, axes = plt.subplots(1, len(frames), figsize=(4 * len(frames), 3.2), sharey=True)
for ax, (t, df) in zip(np.atleast_1d(axes), frames.items()):
    r = df["close"].pct_change().dropna() * 100
    ax.hist(r, bins=120, density=True, alpha=0.7)
    x = np.linspace(-6, 6, 300)
    ax.plot(x, np.exp(-0.5 * ((x - r.mean()) / r.std()) ** 2) / (r.std() * np.sqrt(2 * np.pi)), lw=1.5, label="normal fit")
    ax.set_xlim(-6, 6); ax.set_title(t); ax.set_xlabel("daily return (%)")
np.atleast_1d(axes)[0].legend(); plt.show()

## Autocorrelation: how predictable are daily returns?
Near-zero autocorrelation of returns means past returns carry little *linear* information about the next day's
return, so low classification accuracy should be expected. Absolute returns, by contrast, are usually positively
autocorrelated (volatility clustering), so volatility features carry information about *risk* even when they say little about direction.

In [ ]:
lags = range(1, 11)
ac = {}
for t, df in frames.items():
    r = df["close"].pct_change().dropna()
    ac[(t, "returns")] = [r.autocorr(l) for l in lags]
    ac[(t, "|returns|")] = [r.abs().autocorr(l) for l in lags]
pd.DataFrame(ac, index=pd.Index(lags, name="lag")).round(3)

## Target class distribution (full sample)
The neutral band is ±`neutral_threshold` around zero (configurable in `config.yaml`).

In [ ]:
from src.targets import build_target_frame, class_distribution
out = []
for t, df in frames.items():
    tg = build_target_frame(df, cfg["target"], cfg["execution"]["convention"])
    d = class_distribution(tg["label"], cfg["target"]["scheme"]); d.insert(0, "ticker", t); out.append(d)
pd.concat(out).pivot(index="ticker", columns="class", values="frequency").round(3)